# Stage 1 – Data preparation

Loads the LendingClub accepted and rejected loan files with Polars, keeps completed loans, drops sparse and leakage columns, and saves `accepted_cleaned.csv`.

Part of **Explainable AI for Credit Risk Prediction**. See the main README for the full pipeline.

In [9]:
# 01_data_loading.ipynb
# =====================================
# Stage 1: Data Preparation with Polars
# Project: Explainable AI for Credit Risk Prediction (Sai)
# =====================================

import polars as pl
import os

# -----------------------------------------------------
# 1. Load Accepted Loans Data (RAW)
# -----------------------------------------------------
accepted_path = "../data/accepted_2007_to_2018Q4.csv"

if not os.path.exists(accepted_path):
    raise FileNotFoundError(f"Raw accepted loans file not found at: {accepted_path}")

print("🔹 Loading raw accepted loans data...")
accepted = pl.read_csv(
    accepted_path,
    ignore_errors=True,   # skip bad rows if any
    low_memory=True       # reduce memory pressure
)
print(accepted.head(5))
print("✅ Raw accepted loans dataset loaded.")
print("Shape:", accepted.shape)
print("First 10 columns:", accepted.columns[:10])

# Quick look at loan_status
if "loan_status" not in accepted.columns:
    raise ValueError("Column 'loan_status' not found in accepted dataset. Check the raw file.")
print("\nLoan status value counts (top 20):")
print(accepted["loan_status"].value_counts().head(20))


# -----------------------------------------------------
# 2. Filter to clean outcomes & create binary target
# -----------------------------------------------------
# Keep only final outcomes: Fully Paid vs Charged Off
valid_statuses = ["Fully Paid", "Charged Off"]
accepted = accepted.filter(pl.col("loan_status").is_in(valid_statuses))

# Create binary target:
#   default = 1 if Charged Off, else 0 (Fully Paid)
accepted = accepted.with_columns([
    (pl.col("loan_status") == "Charged Off")
        .cast(pl.Int8)
        .alias("default")
])

print("\n✅ Filtered to final outcomes & created binary target 'default'.")
print("New shape:", accepted.shape)
print("Target distribution:")
print(accepted["default"].value_counts())


# -----------------------------------------------------
# 3. Drop columns with >80% missing values
# -----------------------------------------------------
print("\n🔹 Calculating missing value ratios per column...")

# null_count returns a DataFrame with one row and one column per original column
null_counts_df = accepted.null_count()  # shape: (1, n_cols)
total_rows = accepted.height

# Convert to long format: column name + null_count
null_counts_long = (
    null_counts_df
    .melt(variable_name="column", value_name="null_count")
    .with_columns(
        (pl.col("null_count") / total_rows).alias("missing_ratio")
    )
)

# Columns to drop: missing_ratio > 0.8
drop_sparse = (
    null_counts_long
    .filter(pl.col("missing_ratio") > 0.8)
    ["column"]
    .to_list()
)

print(f"Columns with >80% missing: {len(drop_sparse)}")
if drop_sparse:
    print("Dropping (sparse) columns (first 20 shown):", drop_sparse[:20])
    accepted = accepted.drop(drop_sparse)
else:
    print("No columns exceeded 80% missingness.")

print("Shape after sparse-column drop:", accepted.shape)


# -----------------------------------------------------
# 4. Drop obvious leakage / post-outcome columns
#    (these contain info about repayments AFTER the decision)
# -----------------------------------------------------
leakage_cols = [
    # direct target / default info
    "loan_status",          # original status (we now use 'default')
    "pymnt_plan",

    # time of issue (usually ok, but we can drop to be safe for temporal leakage)
    "issue_d",

    # post-origination payment aggregates (pure leakage)
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt",

    # post-origination dates
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",

    # mostly administrative / not used in modelling
    "url",
    "desc",
    "title",
    "zip_code",
    "addr_state",
    "policy_code",
    "application_type",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
]

existing_leak_cols = [c for c in leakage_cols if c in accepted.columns]
if existing_leak_cols:
    accepted = accepted.drop(existing_leak_cols)
    print("\n✅ Dropped leakage/post-outcome columns:")
    print(existing_leak_cols)
else:
    print("\nℹ️ No leakage columns from the predefined list were found to drop.")

print("Shape after leakage column drop:", accepted.shape)


# -----------------------------------------------------
# 5. Basic Data Quality Checks
# -----------------------------------------------------

# 5.1 Duplicates check (by all columns)
print("\n🔹 Checking for duplicate rows...")
unique_rows = accepted.unique()
has_duplicates = unique_rows.height != accepted.height
print("Duplicates found:", bool(has_duplicates))

# 5.2 Missing values summary (top 10)
print("\n🔹 Top 10 columns by missing ratio after cleaning:")
null_counts_clean = accepted.null_count().melt(
    variable_name="column", value_name="null_count"
).with_columns(
    (pl.col("null_count") / accepted.height).alias("missing_ratio")
).sort("missing_ratio", descending=True)

print(null_counts_clean.head(10))


# -----------------------------------------------------
# 6. Save Cleaned Dataset (two names to avoid confusion)
# -----------------------------------------------------
save_path_1 = "../data/accepted_cleaned.csv"
save_path_2 = "../data/cleanedaccepted.csv"

print("\n💾 Saving cleaned dataset...")

accepted.write_csv(save_path_1)
print(f"✅ Saved cleaned dataset to: {save_path_1}")

accepted.write_csv(save_path_2)
print(f"✅ Saved cleaned dataset to: {save_path_2}")

print("\nFinal cleaned shape:", accepted.shape)
print("Final number of columns:", len(accepted.columns))


# -----------------------------------------------------
# 7. (Optional) Load Rejected Loans Data for later fairness analysis
# -----------------------------------------------------
rejected_path = "../data/rejected_2007_to_2018Q4.csv"
if os.path.exists(rejected_path):
    print("\n🔹 Loading rejected loans data (for future analysis)...")
    rejected = pl.read_csv(rejected_path, ignore_errors=True, low_memory=True)
    print("✅ Rejected loans dataset loaded.")
    print("Shape:", rejected.shape)
    print("Number of columns:", len(rejected.columns))
else:
    print("\n⚠️ Rejected loans file not found at:", rejected_path)
    print("   This is only needed if you plan to analyse fairness between accepted vs rejected.")


🔹 Loading raw accepted loans data...
shape: (5, 151)
┌──────────┬───────────┬───────────┬─────────────┬───┬────────────────┬────────────────┬───────────────┬───────────────┐
│ id       ┆ member_id ┆ loan_amnt ┆ funded_amnt ┆ … ┆ settlement_dat ┆ settlement_amo ┆ settlement_pe ┆ settlement_te │
│ ---      ┆ ---       ┆ ---       ┆ ---         ┆   ┆ e              ┆ unt            ┆ rcentage      ┆ rm            │
│ i64      ┆ str       ┆ f64       ┆ f64         ┆   ┆ ---            ┆ ---            ┆ ---           ┆ ---           │
│          ┆           ┆           ┆             ┆   ┆ str            ┆ f64            ┆ f64           ┆ f64           │
╞══════════╪═══════════╪═══════════╪═════════════╪═══╪════════════════╪════════════════╪═══════════════╪═══════════════╡
│ 68407277 ┆ null      ┆ 3600.0    ┆ 3600.0      ┆ … ┆ null           ┆ null           ┆ null          ┆ null          │
│ 68355089 ┆ null      ┆ 24700.0   ┆ 24700.0     ┆ … ┆ null           ┆ null           ┆ null       

C:\Users\Public\Documents\Wondershare\CreatorTemp\ipykernel_29292\3420977226.py:68: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  null_counts_df



✅ Dropped leakage/post-outcome columns:
['loan_status', 'pymnt_plan', 'issue_d', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'last_pymnt_d', 'last_credit_pull_d', 'url', 'title', 'zip_code', 'addr_state', 'policy_code', 'application_type']
Shape after leakage column drop: (1345310, 93)

🔹 Checking for duplicate rows...
Duplicates found: False

🔹 Top 10 columns by missing ratio after cleaning:
shape: (10, 3)
┌────────────────────────────────┬────────────┬───────────────┐
│ column                         ┆ null_count ┆ missing_ratio │
│ ---                            ┆ ---        ┆ ---           │
│ str                            ┆ u32        ┆ f64           │
╞════════════════════════════════╪════════════╪═══════════════╡
│ mths_since_recent_bc_dlq       ┆ 1026290    ┆ 0.762865      │
│ mths_since_last_major_derog    ┆ 991560     ┆ 0.737049      │
│ mths_since_recent_revol_delinq

C:\Users\Public\Documents\Wondershare\CreatorTemp\ipykernel_29292\3420977226.py:156: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  null_counts_clean = accepted.null_count().melt(


✅ Saved cleaned dataset to: ../data/accepted_cleaned.csv
✅ Saved cleaned dataset to: ../data/cleanedaccepted.csv

Final cleaned shape: (1345310, 93)
Final number of columns: 93

🔹 Loading rejected loans data (for future analysis)...
✅ Rejected loans dataset loaded.
Shape: (27648741, 9)
Number of columns: 9
